# Charge Neutrality Checker using Pymatgen

This notebook demonstrates how to check if chemical compounds are charge-neutral using pymatgen's oxidation state data.

## Installation

First, make sure pymatgen is installed:

In [ ]:
# Uncomment if you need to install pymatgen
# !pip install pymatgen

## Import Libraries

In [3]:
from pymatgen.core import Composition
from pymatgen.core.periodic_table import Element
from typing import Tuple, Dict, List
import itertools

## Main Function: `is_charge_neutral()`

This function checks if a chemical compound is charge-neutral within a given margin.

In [4]:
def is_charge_neutral(formula: str, margin: float = 0.5) -> Tuple[bool, Dict]:
    """
    Check if a chemical compound is charge-neutral within a given margin
    using pymatgen's oxi_state_guesses.
    
    Args:
        formula: Chemical formula (e.g., 'RuO2', 'Fe2O3')
        margin: Acceptable deviation from zero charge (default: 0.5)
    
    Returns:
        Tuple of (is_neutral: bool, info: dict)
        info dict contains:
            - 'composition': parsed composition
            - 'element_amounts': dict of element to amount
            - 'valid_combinations': list of valid oxidation state combinations
            - 'charges': list of total charges for each valid combination
            - 'min_charge': minimum absolute charge found
    """
    try:
        # Parse the formula using pymatgen's Composition
        comp = Composition(formula)
        
        # Get element amounts
        element_amounts = comp.get_el_amt_dict()
        
        # Get possible oxidation states for each element
        elements = list(element_amounts.keys())
        oxi_states_options = []
        
        for element_str in elements:
            element = Element(element_str)
            # Get common oxidation states from pymatgen
            oxi_states = element.common_oxidation_states
            if not oxi_states:
                # If no common oxidation states, try all oxidation states
                oxi_states = T
            if not oxi_states:
                # If still nothing, we can't proceed
                return False, {
                    'composition': str(comp),
                    'element_amounts': element_amounts,
                    'error': f'No oxidation states available for {element_str}',
                    'valid_combinations': [],
                    'charges': [],
                    'min_charge': None
                }
            oxi_states_options.append(list(oxi_states))
        
        # Try all combinations of oxidation states
        valid_combinations = []
        charges = []
        min_abs_charge = float('inf')
        
        for oxi_combo in itertools.product(*oxi_states_options):
            # Calculate total charge for this combination
            total_charge = 0
            for i, element_str in enumerate(elements):
                amount = element_amounts[element_str]
                oxi_state = oxi_combo[i]
                total_charge += amount * oxi_state
            
            charges.append(total_charge)
            
            # Check if this combination is within margin
            abs_charge = abs(total_charge)
            if abs_charge < min_abs_charge:
                min_abs_charge = abs_charge
            
            if abs_charge <= margin:
                combo_dict = {elements[i]: oxi_combo[i] for i in range(len(elements))}
                valid_combinations.append((combo_dict, total_charge))
        
        is_neutral = len(valid_combinations) > 0
        
        info = {
            'composition': str(comp),
            'element_amounts': element_amounts,
            'valid_combinations': [combo for combo, charge in valid_combinations],
            'charges': [charge for combo, charge in valid_combinations],
            'min_charge': min_abs_charge
        }
        
        return is_neutral, info
        
    except Exception as e:
        return False, {
            'error': str(e),
            'composition': None,
            'element_amounts': None,
            'valid_combinations': [],
            'charges': [],
            'min_charge': None
        }

## Example 1: Simple Check for RuO2

In [37]:
is_neutral, info = is_charge_neutral('RuO2')
print(f"Is neutral? {is_neutral}")
print(f"Valid oxidation states: {info['valid_combinations']}")
print(f"Minimum charge: {info['min_charge']:.4f}")

Is neutral? True
Valid oxidation states: [{'Ru': -2, 'O': 1}, {'Ru': 2, 'O': -1}, {'Ru': 4, 'O': -2}]
Minimum charge: 0.0000


## Example 2: Testing with Different Margins

In [4]:
test_compound = 'Fe2O3'
print(f"Testing {test_compound} with different margins:")
for margin in [0.0, 0.5, 1.0]:
    is_neutral, info = is_charge_neutral(test_compound, margin=margin)
    print(f"  Margin {margin}: {is_neutral} (min charge: {info['min_charge']:.4f})")

Testing Fe2O3 with different margins:
  Margin 0.0: True (min charge: 0.0000)
  Margin 0.5: True (min charge: 0.0000)
  Margin 1.0: True (min charge: 0.0000)


## Example 3: Batch Testing Multiple Compounds

In [5]:
compounds = ['RuO2', 'Fe2O3', 'NaCl', 'TiO2', 'MnO2', 'CuO', 'Al2O3', 'H2O']

print("Batch testing compounds:")
print("-" * 60)
for compound in compounds:
    is_neutral, info = is_charge_neutral(compound)
    status = "✓" if is_neutral else "✗"
    print(f"{status} {compound:10s} - Neutral: {str(is_neutral):5s} (min charge: {info['min_charge']:.4f})")

Batch testing compounds:
------------------------------------------------------------
✓ RuO2       - Neutral: True  (min charge: 0.0000)
✓ Fe2O3      - Neutral: True  (min charge: 0.0000)
✓ NaCl       - Neutral: True  (min charge: 0.0000)
✓ TiO2       - Neutral: True  (min charge: 0.0000)
✓ MnO2       - Neutral: True  (min charge: 0.0000)
✓ CuO        - Neutral: True  (min charge: 0.0000)
✓ Al2O3      - Neutral: True  (min charge: 0.0000)
✓ H2O        - Neutral: True  (min charge: 0.0000)


## Example 4: Detailed Analysis

In [6]:
formula = 'RuO2'
is_neutral, info = is_charge_neutral(formula, margin=0.5)

print(f"Detailed analysis of {formula}:")
print("=" * 60)
print(f"Composition: {info['composition']}")
print(f"Element amounts: {info['element_amounts']}")
print(f"Is charge neutral? {is_neutral}")
print(f"Number of valid combinations: {len(info['valid_combinations'])}")
print(f"\nValid oxidation state combinations:")
for i, combo in enumerate(info['valid_combinations'], 1):
    print(f"  Combination {i}: {combo}")
    charge = sum(info['element_amounts'][el] * oxi for el, oxi in combo.items())
    print(f"    Total charge: {charge:.4f}")

Detailed analysis of RuO2:
Composition: Ru1 O2
Element amounts: {'Ru': 1.0, 'O': 2.0}
Is charge neutral? True
Number of valid combinations: 1

Valid oxidation state combinations:
  Combination 1: {'Ru': 4, 'O': -2}
    Total charge: 0.0000


## Example 5: Exploring Oxidation States Directly

In [6]:
# Direct access to pymatgen's oxidation state data
elements = ['Ru', 'O', 'Co', 'Cr', 'Mn', 'Cu', 'Al', 'Ag', 'Ni', 'C', 'P']

print("Oxidation states from pymatgen:")
print("=" * 60)
for elem_symbol in elements:
    elem = Element(elem_symbol)
    print(f"\n{elem.symbol} ({elem.name})")
    print(f"  All oxidation states: {elem.oxidation_states}")
    print(f"  Common oxidation states: {elem.common_oxidation_states}")

Oxidation states from pymatgen:

Ru (Ru)
  All oxidation states: (-2, 1, 2, 3, 4, 5, 6, 7, 8)
  Common oxidation states: (3, 4)

O (O)
  All oxidation states: (-2, -1, 1, 2)
  Common oxidation states: (-2,)

Co (Co)
  All oxidation states: (-1, 1, 2, 3, 4, 5)
  Common oxidation states: (2, 3)

Cr (Cr)
  All oxidation states: (-2, -1, 1, 2, 3, 4, 5, 6)
  Common oxidation states: (3, 6)

Mn (Mn)
  All oxidation states: (-3, -2, -1, 1, 2, 3, 4, 5, 6, 7)
  Common oxidation states: (2, 4, 7)

Cu (Cu)
  All oxidation states: (1, 2, 3, 4)
  Common oxidation states: (2,)

Al (Al)
  All oxidation states: (1, 3)
  Common oxidation states: (3,)

Ag (Ag)
  All oxidation states: (1, 2, 3)
  Common oxidation states: (1,)

Ni (Ni)
  All oxidation states: (-1, 1, 2, 3, 4)
  Common oxidation states: (2,)

C (C)
  All oxidation states: (-4, -3, -2, -1, 1, 2, 3, 4)
  Common oxidation states: (-4, 4)

P (P)
  All oxidation states: (-3, -2, -1, 1, 2, 3, 4, 5)
  Common oxidation states: (-3, 3, 5)


## Example 6: Manual Verification for RuO2

In [34]:
# Manual calculation showing how the checker works
ru = Element('Ni')
o = Element('O')

print("Manual verification for RuO2:")
print("=" * 60)
print(f"Ru common oxidation states: {ru.common_oxidation_states}")
print(f"O common oxidation states: {o.common_oxidation_states}")
print("\nChecking all combinations:")

for ru_ox in ru.common_oxidation_states:
    for o_ox in o.common_oxidation_states:
        charge = 1 * ru_ox + 2 * o_ox
        status = "✓ NEUTRAL" if charge == 0 else f"✗ charge={charge:+d}"
        print(f"  Ru({ru_ox:+d}) + O2({o_ox:+d}×2) = {charge:+d}  {status}")

Manual verification for RuO2:
Ru common oxidation states: (2,)
O common oxidation states: (-2,)

Checking all combinations:
  Ru(+2) + O2(-2×2) = -2  ✗ charge=-2


## Example 7: Filter Only Neutral Compounds

In [9]:
all_compounds = ['RuO2', 'Fe2O3', 'NaCl', 'H2O', 'TiO2', 'MnO2', 'CuO', 'Al2O3']
neutral_compounds = []

for compound in all_compounds:
    is_neutral, _ = is_charge_neutral(compound, margin=0.1)
    if is_neutral:
        neutral_compounds.append(compound)

print(f"Neutral compounds (margin=0.1): {neutral_compounds}")

Neutral compounds (margin=0.1): ['RuO2', 'Fe2O3', 'NaCl', 'H2O', 'TiO2', 'MnO2', 'CuO', 'Al2O3']


## Your Own Tests

Use the cells below to test your own compounds:

In [5]:
# Test your own compound here
your_compound = 'CoCrO4'  # Change this to your compound
is_neutral, info = is_charge_neutral(your_compound, margin=0.5)

print(f"Formula: {your_compound}")
print(f"Is neutral? {is_neutral}")
print(f"Valid combinations: {info['valid_combinations']}")

Formula: CoCrO4
Is neutral? True
Valid combinations: [{'Co': 2, 'Cr': 6, 'O': -2}]
